In [ ]:
import pandas as pd
import scipy.stats as stats
import sys, os
sys.path.append(os.path.abspath(".."))
import config

In [ ]:
config.path_to_evaluation

In [ ]:
metrics = ["NRMSE", "MASE", "sMAPE"]
p_list = ["5000", "12000", "25000", "50000", "100000", "all"]
feats = ['abs_energy','intermittency','mean','median','kurtosis','skewness','standard_deviation','agg_autocorrelation_max','erraticness','agg_linear_trend_slope']
runs = ["run1", "run2", "run3", "run4", "run5"]

In [ ]:
def calculate_1minusKS(target_feat, source_feat):
    similarity = []
    for feature in feats:
                ks_statistic, p_value = stats.ks_2samp(
                    target_feat[feature], source_feat[feature]
                )
                similarity.append(1 - ks_statistic)
    similarity = sum(similarity)
    return similarity

In [ ]:
results = []

for run in runs:   
    path_to_forecasts = f"{config.path_to_data}/{run}/outputs/forecasts/"
    for div in [10, 5, 0]:
        targets_feat = []
        targets_ids =[]
        for p in p_list:
            concat = pd.read_parquet(f"{config.path_to_data}/concat/CONCAT_feat.parquet")
            target = pd.read_parquet(f"{config.path_to_data}/{run}/interim/target_div{div}_sim{p}{run}/target_div{div}_sim{p}{run}.parquet")
            target_ids = target.id_ts.unique().tolist()
            target_feat = concat[concat["id_ts"].isin(target_ids)][feats + ["id_ts"]]
            targets_feat.append(target_feat)
            targets_ids.append(target_ids)
        targets_feat = pd.concat(targets_feat)
        source = pd.read_parquet(f"{config.path_to_data}/{run}/interim/source_div{div}{run}/source_div{div}{run}.parquet")
        source_ids = source.id_ts.unique().tolist()
        source_feat = concat[concat["id_ts"].isin(source_ids)][feats + ["id_ts"]]
        all_feat = pd.concat([targets_feat, source_feat], axis=0)
        feat_std = all_feat.copy()
        feat_std[feats] = (all_feat[feats] - all_feat[feats].mean()) / all_feat[feats].std()
        for i,p in zip(range(len(p_list)), p_list):
            target_feat_std = feat_std[feat_std["id_ts"].isin(targets_ids[i])]
            source_feat_std = feat_std[feat_std["id_ts"].isin(source_ids)]
            ks = calculate_1minusKS(target_feat_std.drop(columns=["id_ts"]), source_feat_std.drop(columns=["id_ts"]))
            results.append({
                "run": run,
                "div": div,
                "p": p,
                "ks": ks
            })
ks = pd.DataFrame(results)
ks.to_parquet(f"{config.path_to_evaluation}/1-KS_similarity.parquet")
        

In [ ]:
ks = pd.read_parquet(f"{config.path_to_evaluation}/1-KS_similarity.parquet")
ks

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

p_order = [5000, 12000, 25000, 50000, 100000, "all"]

ks["p"] = ks["p"].astype(str)

p_order_str = [str(x) for x in p_order]

fig, ax = plt.subplots(figsize=(10, 6))


for (run, div), group in ks.groupby(["run", "div"]):
    group = group.copy()

    group["p"] = pd.Categorical(
        group["p"],
        categories=p_order_str,
        ordered=True
    )

    group = group.sort_values("p")

    ax.plot(
        range(len(group)),
        group["ks"],
        marker="o",
        label=f"{run}, div={div}"
    )

ax.set_xticks(range(len(p_order_str)))
ax.set_xticklabels(p_order_str)

ax.set_xlabel("p")
ax.set_ylabel("1-KS")
ax.set_title("1-KS accross similarity-level p")
ax.grid(True, alpha=0.3)

ax.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()